# Experimento: FT-Transformer, fase 4 (as features do MLP)

**Por que esta fase.** No MLP, a única mudança nos dados que ganhou do padrão foi **marcar os clientes novos**: `tenure_1` sozinha deu +0,014, e o campeão do MLP (`s6_rmsprop_ks_p50`, KS 0,5530 no placar e 0,5463 no teste) usa três features criadas. Até a fase 3, o FT-Transformer só viu o pré-processamento base.

| Feature | Definição |
|---|---|
| `cliente_novo` | 1 se tenure ≤ 6 |
| `delta_preco` | MonthlyCharges − TotalCharges/(tenure+1): o preço atual menos a média já paga (entra no scaler) |
| `tenure_1` | 1 se tenure ≤ 1 (61% de churn no treino) |

**Por que pode ajudar mais aqui.** O tokenizador transforma cada coluna num token com uma reta (`x·W + b`). O `tenure` entra como uma reta, e um corte como "tenure ≤ 1" a rede precisa montar sozinha. Com a feature, esse corte vira um token próprio, que a atenção usa direto.

## Pergunta

**As features do MLP sobem o KS do FT-Transformer?**

| Rótulo | Base | Dados |
|---|---|---|
| `f_trial0_tres` | trial 0 | + as três |
| `f_r_drop_medio_tres` | `r_drop_medio` | + as três |
| `f_r_combo_moderado_tres` | `r_combo_moderado` (o maior KS do FT até aqui, 0,5444) | + as três |
| `f_r_combo_moderado_<feature>` | `r_combo_moderado` | + cada uma sozinha (3 células) |
| `f_r_combo_moderado_tres_sem_over` | `r_combo_moderado` | + as três, sem oversampling (`trial0_sem_over` foi o 2º do placar) |

São 7 células. As bases vêm da fase 3 sem treinar de novo. A parada continua pela perda com paciência 16, porque na fase 3 a parada pelo KS não ajudou o FT.

**Sem vazamento.** As features são calculadas por linha, sem olhar o rótulo nem outros clientes, e o scaler é ajustado só no treino de cada dobra. Cada célula leva dois vereditos pareados: contra o trial 0 e contra a **própria base sem as features**.

## Como executar no Kaggle

1. *File → Import Notebook* → `experimentos/ft_transformer/ft_transformer_fase4.ipynb`, com o dataset anexado.
2. *Add-ons → Secrets:* `GITHUB_TOKEN` e `WANDB_API_KEY`. **Nunca escreva os tokens neste notebook.**
3. *Settings:* Internet **On** e acelerador **GPU T4 x2**.
4. Rode com `MODO_TESTE = True`. Depois, com `False`, use *Save Version → Save & Run All (Commit)*.

# 0. Preparação do ambiente

In [ ]:
# ===== Configuração da execução (edite aqui) =====
REPO_URL = "https://github.com/diegoflyra/churn_project.git"
BRANCH = "diego/ft-transformer"           # branch com o código (precisa estar publicada no GitHub)
RESULTS_REPO_URL = REPO_URL      # onde espelhar os resultados ("" = não enviar ao GitHub)
RESULTS_BRANCH = "resultados"    # branch órfã só de resultados
RUN_NAME = ""                    # vazio = "ft-fase4-AAAAMMDD-HHMM"; fixe um nome para retomar
RESUME_FROM = ""                 # "" | "github" | pasta com os resultados de uma execução anterior
CSV_PATH = "/kaggle/input/datasets/diegolyra/churn-ibm/telco_ibm.csv"   # vazio = procurar no repositório e em /kaggle/input
POR_GPU = 2                      # processos (dobras ao mesmo tempo) em cada GPU; sem GPU, processos na CPU
FASE3_RUN = "ft-fase3-20261115-1751"   # execução da fase 3 na branch de resultados (ou uma pasta local): traz as fases 1 a 3 já medidas
USAR_WANDB = True                # False = não envia nada ao W&B, mesmo com a chave
USAR_GITHUB = True               # False = não espelha os resultados no GitHub, mesmo com o token
MODO_TESTE = False               # True = passada rápida para validar o notebook; os números não servem como resultado
HORAS_LIMITE = 10.5              # depois disso não começa configurações novas e encerra limpo (o Kaggle corta em 12 h)

In [ ]:
import importlib.util, os, shutil, subprocess, sys, time
from pathlib import Path

INICIO = time.time()


def _load_dotenv(path):
    """Tokens em um .env local (GITHUB_TOKEN, WANDB_API_KEY); nunca sobrescreve variáveis já definidas."""
    if os.path.isfile(path):
        for line in open(path, encoding="utf-8"):
            key, sep, value = line.strip().partition("=")
            if sep and not key.startswith("#") and value.strip() and not os.environ.get(key.strip()):
                os.environ[key.strip()] = value.strip().strip("'").strip('"')


def _secret(name):
    value = os.environ.get(name, "").strip()
    if value:
        return value
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:
        pass
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        pass
    return ""


def _is_repo(p):
    return (p / "comum.py").exists() and (p / "experimentos" / "ft_transformer" / "modelo.py").exists()


local = next((p for p in [Path.cwd(), *Path.cwd().parents] if _is_repo(p)), None)
if local:
    REPO_DIR = local
    _load_dotenv(REPO_DIR / ".env")
    GITHUB_TOKEN = _secret("GITHUB_TOKEN")
    print(f"Código: cópia local em {REPO_DIR}")
else:
    GITHUB_TOKEN = _secret("GITHUB_TOKEN")
    REPO_DIR = Path("/tmp/churn_project")
    url = REPO_URL.replace("https://", f"https://x-access-token:{GITHUB_TOKEN}@", 1) if GITHUB_TOKEN else REPO_URL
    print("GitHub: usando GITHUB_TOKEN" if GITHUB_TOKEN else "GitHub: clone sem token")
    shutil.rmtree(REPO_DIR, ignore_errors=True)
    if subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", BRANCH, url, str(REPO_DIR)]).returncode != 0:
        raise RuntimeError(f"Falha ao clonar a branch {BRANCH}. Confira se ela foi publicada; repositório privado exige GITHUB_TOKEN.")
    subprocess.run(["git", "-C", str(REPO_DIR), "remote", "set-url", "origin", REPO_URL])  # o token não fica salvo no clone
    print(f"Código: {REPO_DIR}")
os.chdir(REPO_DIR)
print(subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True).stdout.strip())

# Instala só o que falta: o Kaggle já traz versões compatíveis com a GPU dele
IMPORTS = {"imbalanced-learn": "imblearn", "scikit-learn": "sklearn"}
faltam = [p.strip() for p in open("requirements.txt", encoding="utf-8")
          if p.strip() and importlib.util.find_spec(IMPORTS.get(p.strip(), p.strip())) is None]
if faltam:
    print("Instalando:", ", ".join(faltam))
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *faltam])

In [ ]:
import json

PASTA_FT = REPO_DIR / "experimentos" / "ft_transformer"
sys.path[:0] = [str(REPO_DIR), str(PASTA_FT)]
os.environ["PYTHONPATH"] = os.pathsep.join([str(REPO_DIR), str(PASTA_FT), os.environ.get("PYTHONPATH", "")])  # processos paralelos acham os módulos
os.environ.setdefault("TORCH_DISABLE_NATIVE_JIT", "1")
import fluxo

WORKDIR = "/kaggle/working" if os.path.isdir("/kaggle/working") else "/content" if os.path.isdir("/content") else str(REPO_DIR)
RUN_NAME = RUN_NAME or time.strftime("ft-fase4-%Y%m%d-%H%M")
if MODO_TESTE and not RUN_NAME.startswith("teste-"):
    RUN_NAME = "teste-" + RUN_NAME
# O teste nunca se mistura com a execução real, e rodar localmente não sobrescreve o que está versionado em resultados/
PASTA = os.path.join(WORKDIR, "execucoes", RUN_NAME) if WORKDIR == str(REPO_DIR) else os.path.join(WORKDIR, "resultados_" + RUN_NAME)
os.makedirs(PASTA, exist_ok=True)
os.environ["CHURN_RESULTADOS"] = PASTA
os.environ["CHURN_CSV"] = fluxo.achar_csv(CSV_PATH)
if MODO_TESTE:
    os.environ["CHURN_MAX_EPOCAS"] = "3"          # antes de importar modelo.py, que lê o limite na importação
    print("MODO_TESTE: 3 épocas e duas células da grade. Os números não servem como resultado.")

if not USAR_GITHUB:
    RESULTS_REPO_URL = ""
if RESUME_FROM == "github":
    fluxo.baixar_github(PASTA, RUN_NAME, RESULTS_REPO_URL, GITHUB_TOKEN, RESULTS_BRANCH)
elif RESUME_FROM:
    shutil.copytree(RESUME_FROM, PASTA, dirs_exist_ok=True)


def salvar():
    return fluxo.salvar_github(PASTA, RUN_NAME, RESULTS_REPO_URL, GITHUB_TOKEN, RESULTS_BRANCH)


def continuar():
    return HORAS_LIMITE * 3600 - (time.time() - INICIO) > 0


import matplotlib.pyplot as plt, numpy as np, pandas as pd
from scipy.stats import ks_2samp
import torch
import comum, modelo as M, fase1 as F1, fase2 as F2, fase4 as F4

DISPOSITIVOS = [f"cuda:{i}" for i in range(torch.cuda.device_count())] or ["cpu"]
if DISPOSITIVOS == ["cpu"]:
    POR_GPU = os.cpu_count() or 1                 # sem GPU: um processo por núcleo
pd.set_option("display.max_columns", None, "display.width", 220, "display.max_colwidth", 70, "display.max_rows", 80)
print(f"RUN_NAME = {RUN_NAME} | resultados em {PASTA}")
print(f"CSV: {os.environ['CHURN_CSV']} | dispositivos: {DISPOSITIVOS} x {POR_GPU} processos | até {M.MAX_EPOCAS} épocas")

# O envio ao GitHub é testado antes de qualquer treino
json.dump(dict(run=RUN_NAME, inicio=time.strftime("%Y-%m-%d %H:%M")), open(os.path.join(PASTA, "inicio.json"), "w"))
GITHUB_OK = bool(RESULTS_REPO_URL and GITHUB_TOKEN) and salvar()
if RESULTS_REPO_URL and not GITHUB_OK:
    print("=" * 90 + "\nATENÇÃO: o envio ao GitHub falhou. Troque o GITHUB_TOKEN nos Secrets ou baixe a pasta de resultados no fim.\n" + "=" * 90)
else:
    print("GitHub:", "envio testado e funcionando" if GITHUB_OK else "desligado")

wandb_run = None
wandb_key = _secret("WANDB_API_KEY") if USAR_WANDB else ""
if wandb_key or os.environ.get("WANDB_MODE") == "offline":
    import wandb
    if wandb_key:
        os.environ["WANDB_API_KEY"] = wandb_key
    wandb_run = wandb.init(project="churn-if702", entity=_secret("WANDB_ENTITY") or None, name=RUN_NAME, dir=WORKDIR,
                           group="ft_transformer", config=dict(modelo="ft_transformer", fase=4, dispositivos=DISPOSITIVOS,
                                                               por_gpu=POR_GPU, teste=MODO_TESTE, padrao=M.PADRAO))
else:
    print("W&B desativado (sem chave ou USAR_WANDB = False).")


def ao_registrar(r):
    """Cada configuração medida vai para o W&B e para o GitHub na hora: se o Kaggle cair, nada se perde."""
    if wandb_run is not None:
        v = r.get("vs_trial0") or {}
        wandb_run.log(dict(rotulo=r["rotulo"], ks_placar=r["ks_kfold"], ep=r["ep"], ks_oof=r["ks_oof"], auc_oof=r["auc_oof"],
                           mse_treino=r.get("mse_treino"), mse_val=r.get("mse_val"), epocas=r.get("epocas"),
                           minutos=r.get("tempo_min"), delta_vs_trial0=v.get("delta")))
    salvar()


# As linhas das fases 1 a 3 (com as três bases) entram aqui sem treinar de novo
import tempfile
if FASE3_RUN and not os.path.isfile(os.path.join(PASTA, "placar.jsonl")):
    with tempfile.TemporaryDirectory() as _tmp:
        ok = (shutil.copytree(FASE3_RUN, _tmp, dirs_exist_ok=True) and True) if os.path.isdir(FASE3_RUN) else \
            fluxo.baixar_github(_tmp, FASE3_RUN, RESULTS_REPO_URL or REPO_URL, GITHUB_TOKEN, RESULTS_BRANCH)
        if ok and os.path.isfile(os.path.join(_tmp, "placar.jsonl")):
            shutil.copy(os.path.join(_tmp, "placar.jsonl"), PASTA)
            shutil.copytree(os.path.join(_tmp, "placar_oof"), os.path.join(PASTA, "placar_oof"), dirs_exist_ok=True)
anteriores = {r["rotulo"]: r for r in comum.ler_placar(PASTA) if r["modelo"] == "ft_transformer"}
print(f"Fases 1 a 3: {len(anteriores)} linhas trazidas de {FASE3_RUN}" if anteriores else "Fases 1 a 3: nada trazido; as bases serão medidas aqui")


def medir(itens):
    return F1.garantir(PASTA, itens, DISPOSITIVOS, POR_GPU, continuar=continuar, ao_registrar=ao_registrar)


def mostrar(linhas, campos=("vs_trial0", "vs_base")):
    display(F1.tabela(linhas, campos).round(4))

In [ ]:
# Verificações rápidas antes de gastar tempo de treino
assert comum.assinatura_teste() == "e90ba6fab761f037", "o teste mudou: os números não seriam comparáveis com os outros modelos"
rng = np.random.default_rng(1); y_ = (rng.random(2000) < .265).astype(int); p_ = np.clip(rng.normal(.3 + .25 * y_, .2), 0, 1)
assert abs(comum.ks(y_, p_) - ks_2samp(p_[y_ == 1], p_[y_ == 0]).statistic) < 1e-9
D = F1.divs()
assert len(D) == 15
for d in D:
    assert not set(d["treino"]) & set(d["validacao"]) and not set(d["ajuste"]) & set(d["parada"])
Xtr_, _, ytr_, _ = F1._dados()
Xa_, _, _ = comum.preparar(Xtr_, ytr_, Xtr_.iloc[:5])
n_param = sum(p.numel() for p in M.FTTransformer(Xa_.shape[1], **M.arquitetura(M.PADRAO)).parameters())
print(f"teste com a assinatura oficial | KS conferido contra scipy | 15 divisões sem sobreposição | assinatura do treino {F1.assinatura()}")
print(f"{Xa_.shape[1]} features = {Xa_.shape[1]} tokens + [CLS] | FT-Transformer padrão: {n_param:,} parâmetros para {len(Xtr_):,} clientes")
Xf_, _, _ = comum.preparar(Xtr_, ytr_, Xtr_.iloc[:5], features=F4.TRES)
print(f"com as features do MLP: {Xf_.shape[1]} tokens + [CLS] ({', '.join(F4.TRES)})")

# 1. As features do MLP sobem o KS do FT-Transformer?

Cada célula passa pela régua das fases anteriores (15 dobras, parada nos 20% internos, veredito pareado). Vale olhar:

- **`vs_base`:** o veredito contra a mesma configuração sem as features. É ele que responde à pergunta;
- **as três sozinhas contra juntas**, na base `r_combo_moderado`: qual feature carrega o ganho;
- **`mse_treino` contra `mse_val`:** se os tokens a mais aumentaram o sobreajuste.

In [ ]:
T0 = ("ft_transformer", "trial0")
GRADE = F4.grade_features()
assert all(b in anteriores for b in F4.BASES) or MODO_TESTE, "faltam bases no placar trazido"
if MODO_TESTE:
    GRADE = {r: g for r, g in GRADE.items() if r in ("f_trial0_tres", "f_r_combo_moderado_tenure_1")}
bases = [(b, F4.BASES[b], None if b == "trial0" else {"vs_trial0": T0}) for b in F4.BASES]
itens = bases + [(r, c, {"vs_trial0": T0, "vs_base": ("ft_transformer", b)}) for r, (b, c) in GRADE.items()]
t0 = time.time()
linhas = medir(itens)
print(f"{(time.time() - t0) / 60:.1f} min")
novas = [r for r in linhas if r["rotulo"] in GRADE]
tg = F1.tabela(novas, ("vs_trial0", "vs_base")).drop(columns=["blocos", "d_token"])
meta = pd.DataFrame([dict(rotulo=r, base=b, features=", ".join(c["dados"]["features"])) for r, (b, c) in GRADE.items()])
display(meta.merge(tg, on="rotulo").round(4))

# 2. Placar das fases 1 a 4

In [ ]:
todas = [r for r in comum.ler_placar(PASTA) if r["modelo"] == "ft_transformer"]
tab = F1.tabela(sorted(todas, key=lambda r: -r["ks_kfold"]), ("vs_trial0", "vs_base"))
display(tab.round(4))
tab.to_csv(os.path.join(PASTA, "placar_ft_transformer_fase4.csv"), index=False)
open(os.path.join(PASTA, "PLACAR.md"), "w", encoding="utf-8").write(comum.relatorio_placar(PASTA))
ganharam = [r["rotulo"] for r in todas if (r.get("vs_trial0") or {}).get("veredito") == "ganhou"]
perderam = [r["rotulo"] for r in todas if (r.get("vs_trial0") or {}).get("veredito") == "perdeu"]
print(f"Ganharam do trial 0: {ganharam or 'nenhuma'}\nPerderam: {perderam or 'nenhuma'}")
salvar()
if wandb_run is not None:
    wandb_run.summary.update(dict(ganharam=ganharam, perderam=perderam))
    wandb_run.finish()
print(f"Tempo total: {(time.time() - INICIO) / 3600:.2f} h")
print("Resultados em", PASTA)
print(sorted(os.listdir(PASTA)))

### 📝 Análise da fase 4

- **As três features ganharam da própria base em alguma configuração?** _…_
- **Qual feature carrega o ganho (as três sozinhas contra juntas)?** _…_
- **Sem oversampling, com as features, mudou algo?** _…_
- **O FT chegou perto do MLP (0,5530 no placar)?** _…_
- **O que segue para a próxima fase:** _…_